# 05 · Merge, test and publish

1. Kaggle: **Add-ons → Secrets** → tick `HF_TOKEN` for this notebook.
2. Settings → Accelerator → **GPU T4 x2**, Internet **On**.
3. **Save Version → Save & Run All (Commit)**. Do not use Run All.

Nothing is uploaded to the public repo unless the merged model passes the safety test. All changes to the public repo happen in one single update at the very end.

In [ ]:
!pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel
import os, re, gc, json, time, random, shutil, string, fnmatch, subprocess
from bisect import bisect_right
from collections import Counter
import torch, pandas as pd
from huggingface_hub import HfApi, hf_hub_download, list_repo_files, snapshot_download
assert torch.cuda.is_available(), "No GPU. Kaggle: Settings > Accelerator > GPU T4 x2 (and Internet: On). Colab: Runtime > Change runtime type > T4 GPU"

def get_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

HF_TOKEN = get_token()
assert HF_TOKEN, "HF_TOKEN secret missing. Kaggle: Add-ons > Secrets > add HF_TOKEN and tick it for this notebook."
api = HfApi(token=HF_TOKEN)
PUBLIC_REPO = "CSroseX/Legal_llama3.1_part5"
SAVE_REPO = f"{api.whoami()['name']}/legal-llama-training"
assert "final/adapter_model.safetensors" in api.list_repo_files(SAVE_REPO), f"No trained adapter in {SAVE_REPO}/final"
assert "evaluation/new.json" in api.list_repo_files(SAVE_REPO), "Run the evaluation notebook first"
api.create_branch(PUBLIC_REPO, branch="write-test", exist_ok=True)
api.delete_branch(PUBLIC_REPO, branch="write-test")
ADAPTER_DIR = os.path.join(snapshot_download(SAVE_REPO, allow_patterns=["final/*"], token=HF_TOKEN), "final")
WORK = "/tmp/publish_work"
PUBLISH = f"{WORK}/publish"
BASE_DIR = f"{WORK}/base16"
os.makedirs(PUBLISH, exist_ok=True)
SEED = 3407

def free_gb():
    return round(shutil.disk_usage("/tmp").free / 1e9, 1)

report = {"public_repo": PUBLIC_REPO, "free_disk_gb_at_start": free_gb()}
print("Adapter found, write access to", PUBLIC_REPO, "confirmed. Free disk:", free_gb(), "GB")

In [ ]:
NOT_FOUND = "Not found."
ALPACA = (
    "Below is an instruction that describes a task, paired with an input that provides further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{}\n\n### Input:\n{}\n\n### Response:\n"
)
INSTRUCTION = (
    "{question}\n"
    "Quote the exact text from the contract excerpt that answers this, one passage per line. "
    'If there is none, answer "Not found."'
)
CHAT_TEMPLATE = (
    "{%- for message in messages -%}"
    "{%- if message['role'] == 'user' -%}"
    "{%- set parts = message['content'].split('\\n---\\n', 1) -%}"
    "{{ bos_token }}Below is an instruction that describes a task, paired with an input that provides further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{{ parts[0] }}\n"
    "Quote the exact text from the contract excerpt that answers this, one passage per line. "
    'If there is none, answer "Not found."\n\n'
    "### Input:\n{{ parts[1] if parts | length > 1 else '' }}\n\n### Response:\n"
    "{% elif message['role'] == 'assistant' -%}"
    "{{ message['content'] }}{{ eos_token }}"
    "{%- endif -%}"
    "{%- endfor -%}"
)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from peft import PeftModel

BASE16 = "unsloth/llama-3-8b"
snapshot_download(BASE16, local_dir=BASE_DIR, allow_patterns=["*.json", "*.safetensors"], token=HF_TOKEN)
base = AutoModelForCausalLM.from_pretrained(BASE_DIR, dtype=torch.float16, low_cpu_mem_usage=True, device_map={"": "cpu"})
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
merged.save_pretrained(PUBLISH, safe_serialization=True, max_shard_size="5GB")
del base, merged
gc.collect()
shutil.rmtree(BASE_DIR)

tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
tok.chat_template = CHAT_TEMPLATE
tok.padding_side = "left"
tok.save_pretrained(PUBLISH)
GenerationConfig(
    bos_token_id=tok.bos_token_id,
    eos_token_id=tok.eos_token_id,
    pad_token_id=tok.pad_token_id,
    do_sample=False,
    max_new_tokens=256,
).save_pretrained(PUBLISH)
report["merged_files"] = sorted(os.listdir(PUBLISH))
print(report["merged_files"], "free disk:", free_gb(), "GB")

In [ ]:
q = 'Highlight the parts (if any) of this contract related to "Governing Law" that should be reviewed by a lawyer.'
page = "This Agreement shall be governed by the laws of the State of New York.\n---\nnot a separator issue"
rendered = tok.apply_chat_template([{"role": "user", "content": f"{q}\n---\n{page}"}], tokenize=False, add_generation_prompt=True)
expected = tok.bos_token + ALPACA.format(INSTRUCTION.format(question=q), page)
assert rendered == expected, (rendered, expected)
report["chat_template_matches_training_format"] = True
print(rendered)

In [ ]:
DATA_REPO = "theatticusproject/cuad-qa"
DATA_REV = "refs/convert/parquet"
paths = [
    hf_hub_download(DATA_REPO, f, repo_type="dataset", revision=DATA_REV)
    for f in list_repo_files(DATA_REPO, repo_type="dataset", revision=DATA_REV)
    if f.startswith("default/test/") and f.endswith(".parquet")
]
test = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
test["answer_texts"] = test["answers"].apply(lambda a: list(a["text"]))
test["answer_starts"] = test["answers"].apply(lambda a: [int(s) for s in a["answer_start"]])
test["has_answer"] = test["answer_texts"].apply(len) > 0
test = test.drop(columns=["answers"])

WINDOW, MAX_WINDOW, N_POSITIVE, N_NEGATIVE = 512, 1024, 500, 500
contracts = test.drop_duplicates("title").set_index("title")["context"].to_dict()
token_starts, token_ends = {}, {}
for t, c in contracts.items():
    offsets = tok(c, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
    token_starts[t] = [s for s, _ in offsets]
    token_ends[t] = [e for _, e in offsets]

def token_index(t, char):
    return max(0, bisect_right(token_starts[t], char) - 1)

def answer_spans(row):
    return [(s, s + len(a), a) for a, s in zip(row.answer_texts, row.answer_starts)]

def make(row, start, end):
    a, b = token_starts[row.title][start], token_ends[row.title][end - 1]
    inside = []
    for s, e, text in answer_spans(row):
        if a <= s and e <= b:
            inside.append((s, text))
        elif s < b and a < e:
            return None
    answers = list(dict.fromkeys(text for _, text in sorted(inside)))
    return {"title": row.title, "question": row.question, "input": contracts[row.title][a:b], "output": "\n".join(answers) if answers else NOT_FOUND}

def positive(row, rng):
    t = row.title
    n = len(token_starts[t])
    s, e, _ = answer_spans(row)[0]
    i0, i1 = token_index(t, s), token_index(t, e - 1) + 1
    size = max(WINDOW, i1 - i0 + 64)
    if size > MAX_WINDOW:
        return None
    size = min(size, n)
    lo, hi = max(0, i1 - size), min(i0, n - size)
    start = rng.randint(lo, hi) if lo <= hi else max(0, n - size)
    return make(row, start, start + size)

def negative(row, rng, tries=20):
    n = len(token_starts[row.title])
    size = min(WINDOW, n)
    for _ in range(tries):
        start = rng.randint(0, n - size)
        ex = make(row, start, start + size)
        if ex and ex["output"] == NOT_FOUND:
            return ex
    return None

rng = random.Random(SEED)
answered, unanswered = test[test.has_answer], test[~test.has_answer]
pos = [x for x in (positive(r, rng) for r in answered.sample(frac=1, random_state=SEED).itertuples()) if x][:N_POSITIVE]
neg_pool = (
    list(unanswered.sample(min(N_NEGATIVE, len(unanswered)), random_state=SEED).itertuples())
    + list(answered.sample(min(N_NEGATIVE, len(answered)), random_state=SEED).itertuples())
)
rng.shuffle(neg_pool)
neg = [x for x in (negative(r, rng) for r in neg_pool) if x][:N_NEGATIVE]
exam = pos + neg

saved = json.load(open(hf_hub_download(SAVE_REPO, "evaluation/new.json", token=HF_TOKEN)))["rows"]
assert len(saved) == len(exam) and all(r["gold"] == x["output"] for r, x in zip(saved, exam)), "Exam rebuild does not match the evaluation run"
GATE_IDS = list(range(0, 60)) + list(range(len(pos), len(pos) + 40))
print("Exam rebuilt identically;", len(GATE_IDS), "pages selected for the safety test")

In [ ]:
def norm(s):
    s = "".join(ch for ch in s.lower() if ch not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def word_overlap(pred, gold):
    p, g = norm(pred).split(), norm(gold).split()
    if not p or not g:
        return float(p == g)
    same = sum((Counter(p) & Counter(g)).values())
    if same == 0:
        return 0.0
    precision, recall = same / len(p), same / len(g)
    return 2 * precision * recall / (precision + recall)

def is_not_found(pred):
    return norm(pred).startswith("not found")

def squash(s):
    return " ".join(s.lower().split())

def quoted(pred, page):
    lines = [l.strip().strip('"') for l in pred.split("\n") if l.strip()]
    return bool(lines) and all(squash(l) in squash(page) for l in lines)

def summarize(examples, preds):
    o, nf, q = [], [], []
    for ex, p in zip(examples, preds):
        if ex["output"] == NOT_FOUND:
            nf.append(is_not_found(p))
        else:
            o.append(0.0 if is_not_found(p) else word_overlap(p, " ".join(ex["output"].split("\n"))))
        if not is_not_found(p):
            q.append(quoted(p, ex["input"]))
    return {
        "word_overlap": round(sum(o) / len(o), 3),
        "correct_not_found": round(sum(nf) / len(nf), 3),
        "not_quoted_from_contract": round(1 - sum(q) / len(q), 3) if q else 0.0,
    }

model, gate_tok = FastLanguageModel.from_pretrained(model_name=PUBLISH, max_seq_length=2048, load_in_4bit=True, dtype=None)
FastLanguageModel.for_inference(model)
gate_tok.padding_side = "left"
gate_exam = [exam[i] for i in GATE_IDS]
prompts = [
    gate_tok.apply_chat_template([{"role": "user", "content": f"{x['question']}\n---\n{x['input']}"}], tokenize=False, add_generation_prompt=True)
    for x in gate_exam
]
preds = []
for b in range(0, len(prompts), 8):
    enc = gate_tok(prompts[b : b + 8], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=200, do_sample=False, temperature=None, top_p=None,
                             eos_token_id=gate_tok.eos_token_id, pad_token_id=gate_tok.pad_token_id)
    preds += [gate_tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]

merged_scores = summarize(gate_exam, preds)
adapter_scores = summarize(gate_exam, [saved[i]["pred"] for i in GATE_IDS])
report["safety_test"] = {"merged": merged_scores, "before_merge": adapter_scores}
print(json.dumps(report["safety_test"], indent=1))
del model
gc.collect()
torch.cuda.empty_cache()
assert merged_scores["word_overlap"] >= adapter_scores["word_overlap"] - 0.05, "STOPPED: merged model answers worse than before merging. Nothing was published."
assert merged_scores["correct_not_found"] >= adapter_scores["correct_not_found"] - 0.05, "STOPPED: merged model says Not found less reliably. Nothing was published."
assert merged_scores["not_quoted_from_contract"] <= 0.02, "STOPPED: merged model is not quoting the contract. Nothing was published."
print("SAFETY TEST PASSED")

In [ ]:
LLAMA = f"{WORK}/llama.cpp"
def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

sh(f"git clone --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA}")
sh(f"pip install -q {LLAMA}/gguf-py sentencepiece protobuf")
sh(f"cmake -S {LLAMA} -B {LLAMA}/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null")
sh(f"cmake --build {LLAMA}/build --target llama-quantize -j 4 > /dev/null")
sh(f"python {LLAMA}/convert_hf_to_gguf.py {PUBLISH} --outtype q8_0 --outfile {PUBLISH}/unsloth.Q8_0.gguf")
sh(f"{LLAMA}/build/bin/llama-quantize --allow-requantize {PUBLISH}/unsloth.Q8_0.gguf {PUBLISH}/unsloth.Q4_K_M.gguf Q4_K_M")

from gguf import GGUFReader
report["gguf"] = {}
for name in ["unsloth.Q8_0.gguf", "unsloth.Q4_K_M.gguf"]:
    reader = GGUFReader(f"{PUBLISH}/{name}")
    field = reader.fields["tokenizer.chat_template"]
    template = bytes(field.parts[field.data[0]]).decode()
    assert template == CHAT_TEMPLATE, f"{name}: chat template missing or different"
    report["gguf"][name] = {"size_gb": round(os.path.getsize(f"{PUBLISH}/{name}") / 1e9, 2), "chat_template_inside": True}
print(report["gguf"], "free disk:", free_gb(), "GB")

In [ ]:
REQUIRED = {"config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json", "unsloth.Q8_0.gguf", "unsloth.Q4_K_M.gguf"}
OPTIONAL = {"chat_template.jinja", "special_tokens_map.json", "model.safetensors", "model.safetensors.index.json"}
DELETE_PATTERNS = [
    "*.bin", "pytorch_model.bin.index.json", "model-*.safetensors", "model.safetensors", "model.safetensors.index.json",
    "*.gguf", "special_tokens_map.json", "chat_template.jinja",
]
local = set(os.listdir(PUBLISH))
shards = {f for f in local if re.fullmatch(r"model-\d{5}-of-\d{5}\.safetensors", f)}
extra = local - REQUIRED - OPTIONAL - shards
for f in extra:
    p = os.path.join(PUBLISH, f)
    shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)
local = set(os.listdir(PUBLISH))
has_weights = bool(shards and "model.safetensors.index.json" in local) or "model.safetensors" in local
template_saved = "chat_template.jinja" in local or "chat_template" in json.load(open(f"{PUBLISH}/tokenizer_config.json"))
assert REQUIRED <= local and has_weights and template_saved, "Missing files, nothing was published"

remote = api.list_repo_files(PUBLIC_REPO)
upload_list = sorted(os.listdir(PUBLISH))
delete_list = [f for f in remote if any(fnmatch.fnmatch(f, p) for p in DELETE_PATTERNS) and f not in upload_list]
report["publish_plan"] = {"add_or_replace": upload_list, "delete": delete_list, "removed_local_extras": sorted(extra)}
print(json.dumps(report["publish_plan"], indent=1))

commit = api.upload_folder(
    repo_id=PUBLIC_REPO,
    folder_path=PUBLISH,
    delete_patterns=DELETE_PATTERNS,
    commit_message="Replace with retrained model: chunked CUAD training, saved prompt format, new GGUF files, remove old .bin duplicates",
)
after = sorted(api.list_repo_files(PUBLIC_REPO))
report["published_commit"] = str(commit)
report["repo_files_after"] = after
assert not any(f.endswith(".bin") for f in after)
json.dump(report, open(f"{WORK}/publish_report.json", "w"), indent=1)
api.upload_file(path_or_fileobj=f"{WORK}/publish_report.json", path_in_repo="publish/report.json", repo_id=SAVE_REPO, commit_message="publish report")
print(json.dumps(report, indent=1))